# Анализ данных и признаки
Анализ данных, очистка лога событий и признаки для каждой куки. Модель в отдельном файле.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import roc_auc_score

In [ ]:
date_cols = ['cookie_created_at', 'window_start_ts', 'window_end_ts']
train = pd.read_csv('data/train.csv', parse_dates=date_cols)
test = pd.read_csv('data/test.csv', parse_dates=date_cols)
events = pd.read_csv('data/events.csv.gz', parse_dates=['event_ts'])

print(f'train: {len(train)} кук, test: {len(test)} кук, событий: {len(events)}')
print(f'train: {train.window_start_ts.min():%d.%m} – {train.window_start_ts.max():%d.%m}, '
      f'test: {test.window_start_ts.min():%d.%m} – {test.window_start_ts.max():%d.%m}')
print(f'доля ботов в train: {train.target.mean():.3f}')

## 1. Данные по дням
Test идёт по датам сразу после train, поэтому случайный сплит не подходит. Валидация по времени.

In [ ]:
by_day = train.groupby(train['window_start_ts'].dt.date)['target'].agg(['size', 'mean'])

fig, axes = plt.subplots(1, 2, figsize=(12, 3.5))
axes[0].bar(by_day.index.astype(str), by_day['size'])
axes[0].set_title('Количество кук по дням (train)')
axes[1].plot(by_day.index.astype(str), by_day['mean'], marker='o')
axes[1].axhline(train['target'].mean(), color='grey', linestyle='--')
axes[1].set_title('Доля ботов по дням')
for ax in axes:
    ax.tick_params(axis='x', rotation=60)
plt.tight_layout()
plt.show()

Доля ботов стабильна 6–9% в день, резких скачков нет.

## 2. Проверка лога событий

In [ ]:
cookies = pd.concat([train, test])
ev = events.merge(cookies[['cookie_id', 'window_start_ts', 'window_end_ts', 'target']], on='cookie_id')
ev['group'] = ev['target'].map({0: 'human', 1: 'bot'}).fillna('test')

# события вне окна наблюдения
ev['outside_window'] = (ev['event_ts'] < ev['window_start_ts']) | (ev['event_ts'] >= ev['window_end_ts'])
outside_share = ev.groupby('group')['outside_window'].mean()

outside_share.plot(kind='bar', figsize=(5, 3), title='Доля событий вне окна наблюдения', rot=0)
plt.show()

In [ ]:
# полные дубликаты строк
print('доля дубликатов:', events.duplicated().mean().round(3))

In [ ]:
# идут ли события внутри куки в файле по времени
step_back = events.groupby('cookie_id')['event_ts'].diff().dt.total_seconds() < 0
print('доля шагов назад во времени:', step_back.mean().round(2))

In [ ]:
# варианты написания платформы
print('значения platform:', sorted(events['platform'].unique()))

Выводы:
- Событий вне окна у ботов в train намного больше, чем у людей, а в test их нет совсем. Использовать их нельзя по условию, и на test такой признак всё равно не работал бы.
- Дубликаты и перемешанный порядок строк это особенности логирования, их убираем.
- Одна и та же платформа записана по-разному, приводим к одному виду.

In [ ]:
PLATFORM_MAP = {'desktop': 'web', 'iphone': 'ios'}


def clean_events(events, cookies):
    """Оставляет события внутри окна, убирает дубликаты, сортирует по времени."""
    df = events.merge(cookies[['cookie_id', 'window_start_ts', 'window_end_ts']], on='cookie_id')
    in_window = (df['event_ts'] >= df['window_start_ts']) & (df['event_ts'] < df['window_end_ts'])
    df = df[in_window].drop_duplicates(subset=events.columns)

    df['platform'] = df['platform'].str.lower().replace(PLATFORM_MAP)
    df = df.sort_values(['cookie_id', 'event_ts']).reset_index(drop=True)
    return df.drop(columns=['window_start_ts', 'window_end_ts'])


clean = clean_events(events, cookies)
print(f'событий после очистки: {len(clean)}')

## 3. Признаки

Каждую куку описываем одной строкой признаков:
- тайминги: как быстро и насколько равномерно идут события
- какие действия делает кука
- поиск и разнообразие просмотренного
- курсор (только для web)
- возраст куки

In [ ]:
def make_features(clean, cookies):
    df = clean.copy()

    # интервал до предыдущего события и насколько он отличается от прошлого интервала
    df['gap'] = df.groupby('cookie_id')['event_ts'].diff().dt.total_seconds()
    df['gap_change'] = df.groupby('cookie_id')['gap'].diff().abs()

    df['is_night'] = df['event_ts'].dt.hour < 6
    df['is_web'] = df['platform'] == 'web'
    df['no_pointer_web'] = df['pointer_x'].isna().astype(float).where(df['is_web'])
    df['pointer_x_web'] = df['pointer_x'].where(df['is_web'])                                          # новое
    df['pointer_y_web'] = df['pointer_y'].where(df['is_web'])                                          # новое
    df['location_changed'] = df['item_location'] != df.groupby('cookie_id')['item_location'].shift()   # новое
    for event in ['item_view', 'search_results_view', 'photo_swipe', 'favorite_add', 'contact_phone_show']:
        df[event] = df['event_name'] == event

    features = df.groupby('cookie_id').agg(
        n_events=('event_ts', 'size'),
        gap_median=('gap', 'median'),
        gap_change_median=('gap_change', 'median'),
        night_share=('is_night', 'mean'),
        item_view_share=('item_view', 'mean'),
        search_share=('search_results_view', 'mean'),
        photo_share=('photo_swipe', 'mean'),
        favorite_share=('favorite_add', 'mean'),
        phone_share=('contact_phone_show', 'mean'),
        search_page_mean=('search_page', 'mean'),
        n_items=('item_id', 'nunique'),
        n_categories=('item_category', 'nunique'),
        n_locations=('item_location', 'nunique'),
        is_web=('is_web', 'mean'),
        no_pointer_web=('no_pointer_web', 'mean'),
        n_item_events=('item_id', 'count'),                         # новое
        location_changed_share=('location_changed', 'mean'),        # новое
        pointer_x_std=('pointer_x_web', 'std'),                     # новое
        pointer_y_std=('pointer_y_web', 'std'),                     # новое
        pointer_x_max=('pointer_x_web', 'max'),                     # новое
        pointer_y_max=('pointer_y_web', 'max'),                     # новое
    )
    features['item_repeat_share'] = 1 - features['n_items'] / features['n_item_events']

    result = cookies[['cookie_id', 'window_start_ts', 'cookie_created_at']].merge(
        features, on='cookie_id', how='left')
    result['cookie_age_days'] = (result['window_start_ts'] - result['cookie_created_at']).dt.days
    return result.drop(columns=['window_start_ts', 'cookie_created_at', 'n_item_events'])


X_train = make_features(clean, train)
X_test = make_features(clean, test)
y_train = train['target'].values
feature_cols = [col for col in X_train.columns if col != 'cookie_id']

## 4. Влияние признаков на то, что это бот

Для каждого признака считаем ROC-AUC по отдельности.
0.5 - признак бесполезен; чем дальше от 0.5, тем сильнее (ниже 0.5 - у ботов значение меньше).

In [ ]:
comparison = []
for col in feature_cols:
    known = X_train[col].notna()
    comparison.append({
        'feature': col,
        'human_median': X_train.loc[known & (y_train == 0), col].median(),
        'bot_median': X_train.loc[known & (y_train == 1), col].median(),
        'roc_auc': roc_auc_score(y_train[known], X_train.loc[known, col]),
    })

comparison = pd.DataFrame(comparison)
comparison['strength'] = (comparison['roc_auc'] - 0.5).abs()
comparison = comparison.sort_values('strength', ascending=False)

comparison.plot(x='feature', y='strength', kind='barh', figsize=(7, 5), legend=False,
                title='Сила признака: |ROC-AUC − 0.5|')
plt.gca().invert_yaxis()
plt.show()

comparison.round(3)

Распределения самых сильных признаков у ботов и людей.
Для интервалов и возраста куки используем логарифмическую шкалу т к у них длинный хвост.

In [ ]:
plot_features = ['gap_median', 'gap_change_median', 'pointer_x_std',
                 'pointer_x_max', 'n_locations', 'cookie_age_days']
log_scale = ['gap_median', 'gap_change_median', 'cookie_age_days']

fig, axes = plt.subplots(2, 3, figsize=(14, 7))
for ax, col in zip(axes.flat, plot_features):
    values = X_train[col]
    if col in log_scale:
        values = np.log1p(values)
    bins = np.linspace(values.min(), values.quantile(0.99), 40)
    ax.hist(values[y_train == 0], bins=bins, density=True, alpha=0.5, label='human')
    ax.hist(values[y_train == 1], bins=bins, density=True, alpha=0.5, label='bot')
    ax.set_title(f'log(1 + {col})' if col in log_scale else col)
    ax.legend()
plt.tight_layout()
plt.show()

Выводы:
- Сильнее всего различаются тайминги: у ботов интервалы короче и ровнее.
- Боты глубже листают поиск и смотрят больше городов.
- У web-ботов обычно нет данных курсора, куки у ботов более новые.
- «Человеческих» действий (фото, избранное) у ботов меньше.

In [ ]:
X_train['window_start_ts'] = train['window_start_ts']
X_train['target'] = y_train
X_train.to_csv('train_features.csv', index=False)
X_test.to_csv('test_features.csv', index=False)